In [1]:
import pandas as pd
import os
from pathlib import Path
import re

In [2]:
mast_data=pd.read_csv(Path('../data/mast/aggregated_sim_data.csv'))
mast_data.head()

,simulation,run_id,mcons,ncons,tum_mcons,tum_ncons,tum_div,caf_imperm,p_die,ctl_kill,...,count_tumor,count_necro,count_n_killer,count_ctl,count_pdlp,count_dc,count_stroma,count_treg,day,final_day
0,5,12254,0.004,0.001,0.04,0.02,1.0,1.0,0.2,0.6,...,4,0,100,0,0,100,0,0,0,False
1,5,12254,0.004,0.001,0.04,0.02,1.0,1.0,0.2,0.6,...,18,0,443,13,5,449,0,0,1,False
2,5,12254,0.004,0.001,0.04,0.02,1.0,1.0,0.2,0.6,...,3,1,452,58,22,491,0,0,2,False
3,5,12254,0.004,0.001,0.04,0.02,1.0,1.0,0.2,0.6,...,0,1,488,52,67,492,0,0,3,False
4,5,12254,0.004,0.001,0.04,0.02,1.0,1.0,0.2,0.6,...,1,14,463,87,106,487,0,0,4,False


In [3]:
mast_data.columns.tolist()

['simulation',
 'run_id',
 'mcons',
 'ncons',
 'tum_mcons',
 'tum_ncons',
 'tum_div',
 'caf_imperm',
 'p_die',
 'ctl_kill',
 'ctl_kill_pdlp',
 'inject_cure',
 'count_tumor',
 'count_necro',
 'count_n_killer',
 'count_ctl',
 'count_pdlp',
 'count_dc',
 'count_stroma',
 'count_treg',
 'day',
 'final_day']

In [ ]:
time_col = "day"
input_cols = [
    "count_tumor",
    "count_necro",
    "count_n_killer",
    "count_ctl",
    "count_pdlp",
    "count_dc",
    "count_stroma",
    "count_treg",
]


output_cols = [
    "mcons",
    "ncons",
    "tum_mcons",
    "tum_ncons",
    "tum_div",
    "caf_imperm",
    "p_die",
    "ctl_kill",
    "ctl_kill_pdlp",
    "inject_cure",
]
print(f"Input columns: {len(input_cols)}")
print(f"Output columns: {len(output_cols)}")

Input columns: 8
Output columns: 10


In [5]:
normalized_counts=mast_data[input_cols].div(mast_data[input_cols].sum(axis=1), axis=0)
normalized_counts.describe()

input_col_names = [re.sub(r'^count_', 'norm_', col) for col in input_cols]
normalized_counts.columns = input_col_names
normalized_counts.describe()

,norm_tumor,norm_necro,norm_n_killer,norm_ctl,norm_pdlp,norm_dc,norm_stroma,norm_treg
count,451720.000000,451720.000000,451720.000000,451720.000000,451720.000000,451720.000000,451720.000000,451720.000000
mean,0.014917,0.032794,0.310962,0.077630,0.227893,0.331566,0.002388,0.001851
std,0.010956,0.038051,0.137501,0.045131,0.207962,0.130302,0.007336,0.006117
min,0.000000,0.000000,0.016601,0.000000,0.000000,0.030976,0.000000,0.000000
25%,0.005738,0.000896,0.192694,0.044334,0.020993,0.221706,0.000000,0.000000
50%,0.013911,0.017632,0.318601,0.077371,0.184737,0.347289,0.000000,0.000000
75%,0.021241,0.055775,0.445437,0.110928,0.396913,0.459280,0.000000,0.000000
max,0.103377,0.335211,0.516182,0.196615,0.773559,0.518479,0.081790,0.084708


In [6]:
mast_data[input_col_names] = normalized_counts

In [7]:
transformed_mast_data=mast_data[
    [time_col] + input_col_names + output_cols
]
transformed_mast_data

,day,norm_tumor,norm_necro,norm_n_killer,norm_ctl,norm_pdlp,norm_dc,norm_stroma,norm_treg,mcons,ncons,tum_mcons,tum_ncons,tum_div,caf_imperm,p_die,ctl_kill,ctl_kill_pdlp,inject_cure
0,0,0.019608,0.000000,0.490196,0.000000,0.000000,0.490196,0.0,0.0,0.004,0.001,0.04,0.02,1.0,1.0,0.2,0.6,0.15,0
1,1,0.019397,0.000000,0.477371,0.014009,0.005388,0.483836,0.0,0.0,0.004,0.001,0.04,0.02,1.0,1.0,0.2,0.6,0.15,0
2,2,0.002921,0.000974,0.440117,0.056475,0.021422,0.478092,0.0,0.0,0.004,0.001,0.04,0.02,1.0,1.0,0.2,0.6,0.15,0
3,3,0.000000,0.000909,0.443636,0.047273,0.060909,0.447273,0.0,0.0,0.004,0.001,0.04,0.02,1.0,1.0,0.2,0.6,0.15,0
4,4,0.000864,0.012090,0.399827,0.075130,0.091537,0.420553,0.0,0.0,0.004,0.001,0.04,0.02,1.0,1.0,0.2,0.6,0.15,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
451715,2,0.000000,0.000000,0.466484,0.041209,0.000000,0.492308,0.0,0.0,0.001,0.002,0.01,0.01,4.0,0.5,0.1,0.6,0.30,2
451716,0,0.019608,0.000000,0.490196,0.000000,0.000000,0.490196,0.0,0.0,0.001,0.002,0.01,0.01,4.0,0.5,0.1,0.6,0.30,2
451717,1,0.005025,0.000000,0.503948,0.005743,0.001436,0.483848,0.0,0.0,0.001,0.002,0.01,0.01,4.0,0.5,0.1,0.6,0.30,2
451718,0,0.019608,0.000000,0.490196,0.000000,0.000000,0.490196,0.0,0.0,0.001,0.002,0.01,0.01,4.0,0.5,0.1,0.6,0.30,2


In [9]:
transformed_mast_data[mast_data["final_day"]].to_csv('../data/mast/processed_mast_data.csv', index=False)